In [1]:
import os
import pandas as pd
import bev
import cv2
import numpy as np
from tqdm import tqdm
import json
import time
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [3]:
# Constants

NUM_ROWS = 4096

DATASET_PATH = "../../../dataset"
IMAGES_A_PATH = os.path.join(DATASET_PATH, 'images_A')
IMAGES_B_PATH = os.path.join(DATASET_PATH, 'images_B')
METADATA_PATH = os.path.join(DATASET_PATH, 'metadata_A')

DATAFRAME_PATH = "../../eval"
SAVE_PATH = os.path.join(DATAFRAME_PATH, "pipeline_results.csv")

MODEL_PATH = "../../models/pose_landmarker_full.task"

FOCAL_LENGTH_MM = 35.0
SENSOR_WIDTH_MM = 36.0
IMAGE_HEIGHT_PX = 1080
IMAGE_WIDTH_PX = 1920
BASELINE_M = 0.1

In [4]:
# Load Dataframe

df = pd.read_csv(SAVE_PATH, dtype={'id': str})
df.head(6)

,id,actor,pose,cam_pitch,gt_dist,gt_ang_sep,gt_d_yaw,gt_d_pitch,gt_sw_x,gt_sw_y,...,m2pa_dist,m2pa_ang_sep,m2pa_d_yaw,m2pa_d_pitch,m2pa_sw_x,m2pa_sw_y,m2pa_sw_z,m2pa_sc_x,m2pa_sc_y,m2pa_sc_z
0,00001,BP_Ada_C_1,34.0,-6.287881,354.167297,14.391177,-4.068234,13.838668,-0.968621,0.066612,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00002,BP_Ada_C_1,17.0,-80.392527,839.159817,70.426628,10.291927,-70.265796,-0.335014,-0.175880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00003,BP_Ada_C_1,114.0,-50.585528,716.956094,30.322529,-20.436334,29.535145,-0.863197,0.059908,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00004,BP_Ada_C_1,42.0,-20.623707,357.440408,61.804887,-68.575333,9.502579,-0.472476,0.805154,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00005,BP_Ada_C_1,114.0,-16.400097,540.643029,72.647749,82.989366,63.720576,-0.298245,-0.170291,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,00006,BP_Ada_C_1,92.0,-67.072543,511.871710,35.985887,-115.477473,3.051334,-0.809162,0.306925,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Configure BEV Settings

settings = bev.main.default_settings
settings.mode = 'image'
settings.render_mesh = False  # Skip the 3D mesh creation
settings.show = False         # Skip opening a display window
settings.save_dict = False    # Skip saving .npz files to disk

In [6]:
# Load BEV Model

bev_model = bev.BEV(settings)

Using BEV.
Threshold for positive center detection: 0.08


c:\Users\ExoHorizon\miniconda3\envs\fbv-bev\lib\site-packages\bev\main.py:101: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(self.settings.m

In [7]:
def run_model_pipeline(sample_id, camera_pitch_rad):
    # Load JSON Metadata
    json_path = F"{METADATA_PATH}/{sample_id}A.json"
    with open(json_path, 'r') as file:
        raw_data = json.load(file)

    data = {
        'RS': np.array([raw_data['Right Shoulder Coords'][axis] for axis in ['x', 'y', 'z']]),
        'LS': np.array([raw_data['Left Shoulder Coords'][axis] for axis in ['x', 'y', 'z']]),
        'RT': np.array([raw_data['Right Thigh Coords'][axis] for axis in ['x', 'y', 'z']]),
        'LT': np.array([raw_data['Left Thigh Coords'][axis] for axis in ['x', 'y', 'z']]),
        'RW': np.array([raw_data['Right Wrist Coords'][axis] for axis in ['x', 'y', 'z']]),
    }

    # Get Ground Truth Torso Measurements
    shoulder_midpoint = (data['RS'] + data['LS']) / 2
    thigh_midpoint = (data['RT'] + data['LT']) / 2

    shoulder_width = np.linalg.norm(data['RS'] - data['LS'])
    thigh_width = np.linalg.norm(data['RT'] - data['LT'])
    arm_length = np.linalg.norm(data['RS'] - data['RW'])
    torso_height = np.linalg.norm(shoulder_midpoint - thigh_midpoint)

    # Load Image
    image_A_path = F"{IMAGES_A_PATH}/{sample_id}A.png"
    img = cv2.imread(image_A_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    start_time = time.perf_counter()
    # ------------------------------------

    # Run Inference
    outputs = bev_model(img)
    joints = outputs['joints']
    if hasattr(joints, 'detach'):
        joints =  joints.detach().cpu().numpy()

    if joints is None or joints.size == 0:
        return [np.nan] * 10, 0
    
    #   Get 2D Keypoints
    def get_normalized_2d_keypoints(outputs, person_idx=0):
        cam = outputs['cam'][person_idx]
        if hasattr(cam, 'detach'):
            cam = cam.detach().cpu().numpy()
            
        scale = cam[0]
        trans = cam[1:3]

        joints_3d = outputs['joints'][person_idx]
        if hasattr(joints_3d, 'detach'):
            joints_3d = joints_3d.detach().cpu().numpy()
        
        kp2d_ndc = scale * (joints_3d[:, :2] + trans)

        keypoints_2d_norm = (kp2d_ndc + 1.0) / 2.0
        
        return keypoints_2d_norm

    keypoints_2d = get_normalized_2d_keypoints(outputs)

    # Create Numpy Arrays
    keypoints_3d = joints[0].copy()

    # Get Predicted Torso Measurements

    L_SHOULDER, R_SHOULDER = 16, 17
    L_HIP, R_HIP = 1, 2
    R_WRIST = 21

    model_shoulder_dist = np.linalg.norm(keypoints_3d[L_SHOULDER] - keypoints_3d[R_SHOULDER])*100
    model_arm_length = np.linalg.norm(keypoints_3d[R_WRIST] - keypoints_3d[R_SHOULDER])*100

    model_shoulder_mid = (keypoints_3d[L_SHOULDER] + keypoints_3d[R_SHOULDER]) / 2
    model_hip_mid = (keypoints_3d[L_HIP] + keypoints_3d[R_HIP]) / 2
    model_torso_height = np.linalg.norm(model_shoulder_mid - model_hip_mid)*100
    
    # Compute Scale Factor
    ratios = []
    if model_shoulder_dist > 0: ratios.append(shoulder_width / model_shoulder_dist)
    if model_arm_length > 0: ratios.append(arm_length / model_arm_length)
    if model_torso_height > 0: ratios.append(torso_height / model_torso_height)

    if not ratios:
        return [np.nan] * 10, 0

    scale_factor = np.median(ratios)

    # Scale 3D Keypoints
    keypoints_3d_cm = keypoints_3d * 100 * scale_factor

    # Calculate Pixel Focal Length
    focal_length_px = (FOCAL_LENGTH_MM * IMAGE_WIDTH_PX) / SENSOR_WIDTH_MM

    # Get Pixel Distances
    def get_pix_dist(idx1, idx2):
        p1 = keypoints_2d[idx1] * [IMAGE_WIDTH_PX, IMAGE_HEIGHT_PX]
        p2 = keypoints_2d[idx2] * [IMAGE_WIDTH_PX, IMAGE_HEIGHT_PX]
        return np.linalg.norm(p1 - p2)

    sh_mid_2d = (keypoints_2d[L_SHOULDER] + keypoints_2d[R_SHOULDER]) / 2 * [IMAGE_WIDTH_PX, IMAGE_HEIGHT_PX]
    hi_mid_2d = (keypoints_2d[L_HIP] + keypoints_2d[R_HIP]) / 2 * [IMAGE_WIDTH_PX, IMAGE_HEIGHT_PX]

    pixels_shoulders = get_pix_dist(L_SHOULDER, R_SHOULDER)
    pixels_arm = get_pix_dist(R_SHOULDER, R_WRIST)
    pixels_torso = np.linalg.norm(sh_mid_2d - hi_mid_2d)

    # Calculate Distances
    z_estimates = [
        (np.linalg.norm(keypoints_3d_cm[L_SHOULDER] - keypoints_3d_cm[R_SHOULDER]) * focal_length_px) / pixels_shoulders,
        (np.linalg.norm(keypoints_3d_cm[R_SHOULDER] - keypoints_3d_cm[R_WRIST]) * focal_length_px) / pixels_arm,
        (np.linalg.norm(model_shoulder_mid*100*scale_factor - model_hip_mid*100*scale_factor) * focal_length_px) / pixels_torso
    ]

    final_z_distance_cm = np.median(z_estimates)

    hip_center_2d = (keypoints_2d[L_HIP] + keypoints_2d[R_HIP]) / 2
    pixel_x_offset = (hip_center_2d[0] * IMAGE_WIDTH_PX) - (IMAGE_WIDTH_PX / 2)
    pixel_y_offset = (hip_center_2d[1] * IMAGE_HEIGHT_PX) - (IMAGE_HEIGHT_PX / 2)

    final_x_distance_cm = (pixel_x_offset * final_z_distance_cm) / focal_length_px
    final_y_distance_cm = (pixel_y_offset * final_z_distance_cm) / focal_length_px


    # Transform To Camera Coordinates
    transformed_keypoints_3d = keypoints_3d_cm.copy()
    transformed_keypoints_3d[:, 0] += final_x_distance_cm
    transformed_keypoints_3d[:, 1] += final_y_distance_cm
    transformed_keypoints_3d[:, 2] += final_z_distance_cm
    transformed_keypoints_3d /= 100

    # ------------------------------------
    inference_time = time.perf_counter() - start_time

    # Apply Coordinate Conversion
    def convert_to_unreal_coords(points_3d_meters):
        unreal_pts = np.zeros_like(points_3d_meters)
        unreal_pts[:, 0] = points_3d_meters[:, 2] * 100
        unreal_pts[:, 1] = points_3d_meters[:, 0] * 100
        unreal_pts[:, 2] = -points_3d_meters[:, 1] * 100
        return unreal_pts

    ue_keypoints_3d = convert_to_unreal_coords(transformed_keypoints_3d)

    # Given Constants
    camera_coords = np.array([0, 0, 0])
    world_up = np.array([0, 0, 1])

    # Get Right Shoulder & Wrist Coordinates
    shoulder_coords = ue_keypoints_3d[17]
    wrist_coords = ue_keypoints_3d[21]

    # Shoulder-Camera Distance
    distance = np.linalg.norm(shoulder_coords)

    # Shoulder-Wrist Shoulder-Camera Vectors
    shoulder_wrist = wrist_coords - shoulder_coords
    shoulder_wrist /= np.linalg.norm(shoulder_wrist)

    shoulder_camera = camera_coords - shoulder_coords
    shoulder_camera /= np.linalg.norm(shoulder_camera)

    # Angular Separation
    angular_separation_rad = np.arccos(np.clip(np.dot(shoulder_wrist, shoulder_camera), -1.0, 1.0))
    angular_separation_deg = np.rad2deg(angular_separation_rad)

    # Camera Un-Pitch Matrix (Aligns Z-Axis with Gravity)
    c, s = np.cos(-camera_pitch_rad), np.sin(-camera_pitch_rad)
    unpitch_matrix = np.array([
        [c,  0, s],
        [0,  1, 0],
        [-s, 0, c]
    ])

    # Un-Pitch Vectors
    shoulder_wrist_gravity = unpitch_matrix @ shoulder_wrist
    shoulder_wrist_gravity /= np.linalg.norm(shoulder_wrist_gravity)

    shoulder_camera_gravity = unpitch_matrix @ shoulder_camera
    shoulder_camera_gravity /= np.linalg.norm(shoulder_camera_gravity)

    # Yaw & Pitch Components
    shoulder_wrist_gravity_yaw = np.rad2deg(np.atan2(shoulder_wrist_gravity[1], shoulder_wrist_gravity[0]))
    shoulder_wrist_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_wrist_gravity[-1], -1.0, 1.0)))

    shoulder_camera_gravity_yaw = np.rad2deg(np.atan2(shoulder_camera_gravity[1], shoulder_camera_gravity[0]))
    shoulder_camera_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_camera_gravity[-1], -1.0, 1.0)))

    delta_yaw = (shoulder_wrist_gravity_yaw - shoulder_camera_gravity_yaw + 180) % 360 - 180
    delta_pitch = shoulder_wrist_gravity_pitch - shoulder_camera_gravity_pitch

    # Relevant Outputs
    return distance, angular_separation_deg, delta_yaw, delta_pitch, \
           shoulder_wrist[0], shoulder_wrist[1], shoulder_wrist[2], \
           shoulder_camera[0], shoulder_camera[1], shoulder_camera[2], \
           inference_time

In [8]:
# Execution Loop

active_pipeline = 'b3ad' 
pipeline_cols = [f'{active_pipeline}_{m}' for m in ['dist', 'ang_sep', 'd_yaw', 'd_pitch', 'sw_x', 'sw_y', 'sw_z', 'sc_x', 'sc_y', 'sc_z']]

total_inference_seconds = 0.0
successful_counts = 0

print(f"Running Inference for Pipeline: {active_pipeline}")

for i in tqdm(range(len(df)), desc=f"Processing {active_pipeline}"):
    row_id = df.at[i, 'id']
    
    pitch_deg = df.at[i, 'cam_pitch']
    pitch_rad = np.deg2rad(pitch_deg)
    
    try:
        *results, elapsed = run_model_pipeline(row_id, pitch_rad)
        
        df.loc[i, pipeline_cols] = results
        
        total_inference_seconds += elapsed
        successful_counts += 1

    except Exception as e:
        # tqdm.write(f"Pipeline {active_pipeline} failed for ID {row_id}: {e}")
        continue

print(f"\nTotal Cumulative Inference Time: {total_inference_seconds:.4f} seconds")
if successful_counts > 0:
    print(f"Average per sample: {(total_inference_seconds / successful_counts) * 1000:.2f} ms")

Running Inference for Pipeline: b3ad


Processing b3ad:   8%|▊         | 337/4096 [00:32<06:02, 10.38it/s]

No person detected!


Processing b3ad:  10%|▉         | 409/4096 [00:38<05:45, 10.68it/s]

No person detected!


...

No person detected!


Processing b3ad:  99%|█████████▉| 4071/4096 [06:25<00:02, 10.95it/s]

No person detected!


Processing b3ad: 100%|██████████| 4096/4096 [06:28<00:00, 10.56it/s]


Total Cumulative Inference Time: 200.6276 seconds
Average per sample: 52.27 ms


In [8]:
print(df.info(verbose=True, show_counts=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4096 entries, 0 to 4095
Data columns (total 84 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            4096 non-null   object 
 1   actor         4096 non-null   object 
 2   pose          4096 non-null   float64
 3   cam_pitch     4096 non-null   float64
 4   gt_dist       4096 non-null   float64
 5   gt_ang_sep    4096 non-null   float64
 6   gt_d_yaw      4096 non-null   float64
 7   gt_d_pitch    4096 non-null   float64
 8   gt_sw_x       4096 non-null   float64
 9   gt_sw_y       4096 non-null   float64
 10  gt_sw_z       4096 non-null   float64
 11  gt_sc_x       4096 non-null   float64
 12  gt_sc_y       4096 non-null   float64
 13  gt_sc_z       4096 non-null   float64
 14  b3ad_dist     3937 non-null   float64
 15  b3ad_ang_sep  3937 non-null   float64
 16  b3ad_d_yaw    3937 non-null   float64
 17  b3ad_d_pitch  3937 non-null   float64
 18  b3ad_sw_x     3937 non-null 

In [9]:
pd.set_option('display.max_columns', None)
df.head(10)

,id,actor,pose,cam_pitch,gt_dist,gt_ang_sep,gt_d_yaw,gt_d_pitch,gt_sw_x,gt_sw_y,gt_sw_z,gt_sc_x,gt_sc_y,gt_sc_z,b3ad_dist,b3ad_ang_sep,b3ad_d_yaw,b3ad_d_pitch,b3ad_sw_x,b3ad_sw_y,b3ad_sw_z,b3ad_sc_x,b3ad_sc_y,b3ad_sc_z,b3sd_dist,b3sd_ang_sep,b3sd_d_yaw,b3sd_d_pitch,b3sd_sw_x,b3sd_sw_y,b3sd_sw_z,b3sd_sc_x,b3sd_sc_y,b3sd_sc_z,m3ad_dist,m3ad_ang_sep,m3ad_d_yaw,m3ad_d_pitch,m3ad_sw_x,m3ad_sw_y,m3ad_sw_z,m3ad_sc_x,m3ad_sc_y,m3ad_sc_z,m3sd_dist,m3sd_ang_sep,m3sd_d_yaw,m3sd_d_pitch,m3sd_sw_x,m3sd_sw_y,m3sd_sw_z,m3sd_sc_x,m3sd_sc_y,m3sd_sc_z,m2as_dist,m2as_ang_sep,m2as_d_yaw,m2as_d_pitch,m2as_sw_x,m2as_sw_y,m2as_sw_z,m2as_sc_x,m2as_sc_y,m2as_sc_z,m2st_dist,m2st_ang_sep,m2st_d_yaw,m2st_d_pitch,m2st_sw_x,m2st_sw_y,m2st_sw_z,m2st_sc_x,m2st_sc_y,m2st_sc_z,m2pa_dist,m2pa_ang_sep,m2pa_d_yaw,m2pa_d_pitch,m2pa_sw_x,m2pa_sw_y,m2pa_sw_z,m2pa_sc_x,m2pa_sc_y,m2pa_sc_z
0,00001,BP_Ada_C_1,34.0,-6.287881,354.167297,14.391177,-4.068234,13.838668,-0.968621,0.066612,0.239448,-1.0,8.827433e-16,2.006235e-17,585.174945,39.345997,-29.096792,28.461256,-0.800313,0.388789,0.456446,-0.999042,-0.027048,-0.034394,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,637.021029,14.850775,-15.034202,0.433168,-0.969154,0.240048,0.055837,-0.998849,-0.016484,0.045054,339.393950,14.267933,-14.078275,3.014588,-0.969154,0.240048,0.055837,-1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00002,BP_Ada_C_1,17.0,-80.392527,839.159817,70.426628,10.291927,-70.265796,-0.335014,-0.175880,-0.925652,-1.0,-1.058414e-17,3.386925e-17,1913.251864,109.699960,11.003982,-109.546262,0.327019,-0.159929,-0.931387,-0.999943,0.001403,0.010596,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00003,BP_Ada_C_1,114.0,-50.585528,716.956094,30.322529,-20.436334,29.535145,-0.863197,0.059908,0.501300,-1.0,2.477637e-18,-7.928438e-17,1704.993278,169.140734,-166.090185,-94.061589,0.979394,0.180510,0.090572,-0.999849,-0.008058,-0.015365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00004,BP_Ada_C_1,42.0,-20.623707,357.440408,61.804887,-68.575333,9.502579,-0.472476,0.805154,0.358460,-1.0,3.975727e-17,3.975727e-17,671.772309,62.328035,-67.859639,8.661009,-0.493566,0.814348,0.305337,-0.999181,-0.022767,-0.033448,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,347.132425,83.777625,-86.829968,-9.904850,-0.116742,0.983634,0.137240,-0.999842,-0.006147,-0.016684,353.870463,83.295862,-86.455927,-10.861209,-0.116742,0.983634,0.137240,-1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00005,BP_Ada_C_1,114.0,-16.400097,540.643029,72.647749,82.989366,63.720576,-0.298245,-0.170291,0.939176,-1.0,1.577106e-16,1.051404e-16,738.493714,92.781332,119.979303,44.300243,0.017631,-0.451204,0.892247,-0.999042,-0.013678,-0.041560,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,542.156821,59.253452,50.104056,48.555979,-0.519870,-0.336814,0.785043,-0.999870,-0.007647,-0.014186,533.333349,58.676487,50.558918,47.742675,-0.519870,-0.336814,0.785043,-1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,00006,BP_Ada_C_1,92.0,-67.072543,511.871710,35.985887,-115.477473,3.051334,-0.809162,0.306925,0.501053,-1.0,-2.082190e-17,-5.552506e-17,1534.161531,63.127304,-99.797870,-32.114511,-0.457691,0.817597,0.349363,-0.999826,0.001096,-0.018599,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1256.007345,115.512248,-137.822760,-74.508567,0.412396,0.685860,0.599605,-0.999785,-0.012130,-0.016806,504.504538,114.355451,-136.107103,-75.481126,0.412396,0.685860,

In [10]:
# Save Dataframe

df.to_csv(SAVE_PATH, index=False)

In [11]:
# Sanity Check

check_df = pd.read_csv(SAVE_PATH, dtype={'id': str})
print(check_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4096 entries, 0 to 4095
Data columns (total 84 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            4096 non-null   object 
 1   actor         4096 non-null   object 
 2   pose          4096 non-null   float64
 3   cam_pitch     4096 non-null   float64
 4   gt_dist       4096 non-null   float64
 5   gt_ang_sep    4096 non-null   float64
 6   gt_d_yaw      4096 non-null   float64
 7   gt_d_pitch    4096 non-null   float64
 8   gt_sw_x       4096 non-null   float64
 9   gt_sw_y       4096 non-null   float64
 10  gt_sw_z       4096 non-null   float64
 11  gt_sc_x       4096 non-null   float64
 12  gt_sc_y       4096 non-null   float64
 13  gt_sc_z       4096 non-null   float64
 14  b3ad_dist     3981 non-null   float64
 15  b3ad_ang_sep  3981 non-null   float64
 16  b3ad_d_yaw    3981 non-null   float64
 17  b3ad_d_pitch  3981 non-null   float64
 18  b3ad_sw_x     3981 non-null 